# PDF Keyword + Appeal-Scope Harvester (ET Corpus → _Matches)

This script scans a large folder of ET-case PDFs and builds a **targeted "matches" library** for manual review and downstream analysis.

---

## Purpose

Turn a large, unstructured PDF corpus into a structured shortlist of potentially relevant cases using deterministic keyword and regex filtering.

This is a **high-speed recall harvester**, not a semantic or LLM-based precision layer.

---

## Step 1 — Crawl and Pre-Filter PDFs

* Recursively finds all `*.pdf` files under `INPUT_ROOT`.
* Rejects small documents (`pages < MIN_PAGES`).
* Extracts text from the first `TEXT_PAGES_TO_SCAN` pages (cheap front-scan).

This keeps processing fast while avoiding irrelevant small files.

---

## Step 2 — Two-Tier Matching Logic

### Gate Condition (Mandatory)

Every PDF must contain **all phrases in `NEEDLES_ALL`**.

If this fails → the document is ignored.

### Keep Condition (At Least One Required)

After passing the gate, the PDF must satisfy **at least one** of the following:

* Contain any phrase from `NEEDLES_ANY` (simple substring match), OR
* Match `APPEAL_SCOPE_REGEX` (detects appeal-scope limitation language such as:

  * "not raised in the appeal"
  * "outside the scope of the appeal"
  * "declined to consider"
  * "failed to engage with"
  * etc.)

Only documents passing Gate + Keep are retained.

---

## Step 3 — Parallel Scanning

* Uses `ProcessPoolExecutor` with up to `MAX_WORKERS` processes.
* Each PDF is scanned independently.

For matched files, metadata collected includes:

* `path`
* `pages`
* `size_mb`
* `mtime`
* `hit_all`
* `hit_any`
* `appeal_scope_hit`
* `appeal_scope_match` (small snippet of matched phrase)

---

## Step 4 — Structured Copy to Matches Folder

All matched PDFs are copied into `MATCHES_ROOT`.

### Folder Structure

* One folder per `NEEDLES_ANY` term (slugified)
* One dedicated folder `_APPEAL_SCOPE_REGEX` for all regex hits

If `PRESERVE_STRUCTURE = True`, original directory hierarchy is preserved under each subfolder.

This prevents filename collisions and preserves provenance.

---

## Step 5 — Master Index CSV

Exactly one CSV is written to:

`MATCHES_ROOT/_matches_index.csv`

The CSV contains:

* File metadata
* Raw hit indicators
* Boolean columns per NEEDLES_ANY (e.g., `has__predetermination`)
* Boolean column for regex hits (`has__appeal_scope_regex`)
* Root path reference

---

## Output Artifacts

1. Curated PDF library under `MATCHES_ROOT/`
2. Single master index CSV for analysis

---

## Architectural Position

This module is a **deterministic recall harvester**.

It does not perform:

* Semantic analysis
* LLM reasoning
* Paragraph-level anchoring

Its role is to reduce a massive PDF corpus into a manageable, topic-filtered working set for deeper analysis (e.g., Moltie or WBerious precision layers).

---

## Guiding Principle

Fast recall first. Precision and reasoning later.


In [ ]:
# =========================================================
# JN TEST CELL — corpus_builder (JSON-driven, dynamic buckets)
# =========================================================

import sys
import json
import re
from pathlib import Path

import pandas as pd

# =========================================================
# 0) SELECT MODE
# =========================================================
# Options: "offensive" or "defensive"
NEEDLE_MODE = "offensive"

# =========================================================
# 1) Import engine
# =========================================================
ADAPTER_DIR = Path("/home/hello/Projects/Statements/code/adapter").resolve()
if str(ADAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(ADAPTER_DIR))

from corpus_builder import run_corpus_builder

# =========================================================
# 2) Paths
# =========================================================

BASE_MATCHES_ROOT = Path(r"/media/hello/Vault/Tribunals/_Matches").resolve()
MATCHES_ROOT = (BASE_MATCHES_ROOT / NEEDLE_MODE).resolve()
INPUT_ROOT = Path(r"/media/hello/Vault/Tribunals/ET_Cases/").resolve()

NEEDLES_DIR = Path("/home/hello/Projects/Statements/input/needles").resolve()
PACK_PATH = NEEDLES_DIR / f"{NEEDLE_MODE}_needles.json"

# =========================================================
# 3) Load JSON needle pack
# =========================================================
with open(PACK_PATH, "r", encoding="utf-8") as f:
    needle_pack = json.load(f)

NEEDLES_ALL = needle_pack.get("needles_all", [])
NEEDLES_ANY = needle_pack.get("needles_any", [])
REGEX_BUCKETS_RAW = needle_pack.get("regex_buckets", {})

print(f"[mode] {NEEDLE_MODE}")
print(f"[needles_all] {NEEDLES_ALL}")
print(f"[needles_any] {NEEDLES_ANY}")
print(f"[regex_buckets] {list(REGEX_BUCKETS_RAW.keys())}")

# =========================================================
# 4) Compile regex buckets dynamically
# =========================================================
FLAG_MAP = {
    "IGNORECASE": re.IGNORECASE,
    "VERBOSE": re.VERBOSE,
    "MULTILINE": re.MULTILINE,
    "DOTALL": re.DOTALL,
}

compiled_regex_buckets = {}
regex_folder_map = {}

for bucket_name, bucket_cfg in REGEX_BUCKETS_RAW.items():
    flags = 0
    for f_name in bucket_cfg.get("flags", []):
        flags |= FLAG_MAP.get(f_name, 0)

    compiled_regex_buckets[bucket_name] = re.compile(
        bucket_cfg["pattern"],
        flags,
    )

    regex_folder_map[bucket_name] = bucket_cfg.get(
        "folder_name",
        f"_{bucket_name.upper()}",
    )

# =========================================================
# 5) Run corpus builder (NEW ENGINE)
# =========================================================
df = run_corpus_builder(
    input_root=INPUT_ROOT,
    matches_root=MATCHES_ROOT,
    needles_all=NEEDLES_ALL,
    needles_any=NEEDLES_ANY,
    regex_buckets=compiled_regex_buckets,
    regex_folder_map=regex_folder_map,

    cfg_overrides=dict(
        case_sensitive=False,
        min_pages=4,

        # full scan (fast enough with your setup)
        text_pages_head=100000,
        text_pages_tail=0,

        max_workers=24,
        submit_chunk_size=2000,
        preserve_structure=True,
        master_csv_name="_matches_index.csv",

        regex_only_folder_name=needle_pack.get(
            "regex_only_folder_name",
            "_REGEX_ONLY"
        ),
    ),
)

# =========================================================
# 6) Sanity checks
# =========================================================
print("\n[done] df shape:", df.shape)

print("\n[columns]")
print(df.columns.tolist())

print("\n[head]")
display(df.head(10))

if not df.empty:
    ok = df["ok"].fillna(False).astype(bool) if "ok" in df.columns else pd.Series(False, index=df.index)
    err = df["error"].fillna(False).astype(bool) if "error" in df.columns else pd.Series(False, index=df.index)

    print("\n[counts]")
    print("ok:", int(ok.sum()), "| error:", int(err.sum()), "| total rows kept:", len(df))

    if "has__any_needle" in df.columns:
        print("has__any_needle:", int(df["has__any_needle"].fillna(False).astype(bool).sum()))

    # =====================================================
    # Dynamic regex + needle columns
    # =====================================================
    regex_cols = [c for c in df.columns if c.startswith("has__") or "regex" in c.lower()]

    print("\n[match columns]")
    print(regex_cols)

    for c in regex_cols:
        try:
            vals = df[c].fillna(False)
            if vals.isin([True, False]).all():
                print(f"{c}: {int(vals.astype(bool).sum())}")
        except Exception:
            pass

In [ ]:
import sys, re, json
from pathlib import Path
import pandas as pd

# =========================================================
# MODE / PATHS
# =========================================================
# Options: "offensive" or "defensive"
NEEDLE_MODE = "offensive"

# True  -> compile _tag_spec.json from offensive/defensive json
# False -> use existing _tag_spec.json
GENERATE_SPEC = True

# Match artifacts / PDFs / generated tag spec live here
BASE_MATCHES_ROOT = Path("/media/hello/Vault/Tribunals/_Matches").resolve()
MATCHES_ROOT = (BASE_MATCHES_ROOT / NEEDLE_MODE).resolve()
MATCHES_ROOT.mkdir(parents=True, exist_ok=True)

# Source needle jsons live here
NEEDLES_INPUT_ROOT = Path("/home/hello/Projects/Statements/input/needles").resolve()

NEEDLE_JSON_BY_MODE = {
    "offensive": (NEEDLES_INPUT_ROOT / "offensive_needles.json").resolve(),
    "defensive": (NEEDLES_INPUT_ROOT / "defensive_needles.json").resolve(),
}

NEEDLE_SOURCE_JSON = NEEDLE_JSON_BY_MODE.get(NEEDLE_MODE)
if NEEDLE_SOURCE_JSON is None:
    raise ValueError(f"Unsupported NEEDLE_MODE: {NEEDLE_MODE}")

# Generated / loaded runtime spec always lives with the match folder
TAG_SPEC_PATH = (MATCHES_ROOT / "_tag_spec.json").resolve()

# =========================================================
# import engine
# =========================================================
ADAPTER_DIR = Path("/home/hello/Projects/Statements/code/adapter").resolve()
if str(ADAPTER_DIR) not in sys.path:
    sys.path.insert(0, str(ADAPTER_DIR))

from y_runner_engine import YRunnerConfig, run_y_pipeline, canonicalize_tag

# =========================================================
# CALIBRATION INPUTS (OWNED BY JN)
# =========================================================
CSV_PATH = Path("/home/hello/Projects/Statements/input/Leonardo_WS_copy.csv")
TEXT_COL = "text_verbatim"

OUT_DIR = Path("/home/hello/Projects/Statements/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# WS enhanced CSV is mode-specific (different has__ columns per mode)
OUT_ENHANCED_CSV = OUT_DIR / f"Leonardo_WS_enhanced_{NEEDLE_MODE}.csv"

# Y inference is mode-independent (y_spec.py has no mode parameter)
OUT_Y_JSON = OUT_DIR / "Y_inferred.json"

Y_SPEC_PY = Path("/home/hello/Projects/Statements/code/moltie/schemas/y_spec.py")
MODEL = "mistral-small3.2:latest"

DEBUG = False
DEBUG_SLICE = "3"

NEEDLE_TIMEOUT = 180
Y_SPEC_TIMEOUT = 180

# =========================================================
# AUTO / LOAD TAG SPEC
# =========================================================
TAG_SPEC_PATH = (MATCHES_ROOT / "_tag_spec.json").resolve()

def load_tag_spec(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"TagSpec not found at: {path}\n"
            f"Run the corpus_builder cell first to generate '_tag_spec.json', "
            f"or set GENERATE_SPEC=True."
        )
    return json.loads(path.read_text(encoding="utf-8"))

def canonicalize_tag_local(tag: str) -> str:
    s = str(tag).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def build_desc_for_keyword_tag(tag: str) -> str:
    return f"keyword/phrase tag: '{tag}'"

def build_desc_for_regex_tag(tag: str) -> str:
    return f"regex bucket: {tag.replace('_', ' ')}"

def compile_tag_spec_from_needles_json(
    needles_json_path: Path,
    mode: str,
    out_path: Path,
) -> dict:
    if not needles_json_path.exists():
        raise FileNotFoundError(
            f"Needle source json not found: {needles_json_path}"
        )

    payload = json.loads(needles_json_path.read_text(encoding="utf-8"))

    needles_all = payload.get("needles_all", [])
    needles_any = payload.get("needles_any", [])
    regex_buckets = payload.get("regex_buckets", {})

    if not isinstance(needles_all, list):
        raise ValueError("'needles_all' must be a list")
    if not isinstance(needles_any, list):
        raise ValueError("'needles_any' must be a list")
    if not isinstance(regex_buckets, dict):
        raise ValueError("'regex_buckets' must be a dict")

    reserved = {"any_needle", "none"}
    tags = []
    seen = set()

    for raw_tag in needles_any:
        if not isinstance(raw_tag, str):
            continue
        tag = canonicalize_tag_local(raw_tag)
        if not tag or tag in reserved or tag in seen:
            continue
        seen.add(tag)
        tags.append({
            "tag": tag,
            "type": "needle_any",
            "source": {"phrase": raw_tag},
            "desc": build_desc_for_keyword_tag(raw_tag),
            "corpus_column": f"has__{tag}",
        })

    for bucket_name, bucket_meta in regex_buckets.items():
        if not isinstance(bucket_name, str):
            continue
        if not isinstance(bucket_meta, dict):
            continue
        if not bucket_meta.get("pattern"):
            continue

        tag = canonicalize_tag_local(bucket_name)
        if not tag or tag in reserved or tag in seen:
            continue
        seen.add(tag)
        tags.append({
            "tag": tag,
            "type": "regex",
            "source": {"name": bucket_name},
            "desc": build_desc_for_regex_tag(bucket_name),
            "corpus_column": f"has__{tag}",
        })

    spec = {
        "version": "v1",
        "compiled_from": str(needles_json_path),
        "compiled_mode": mode,
        "needles_all_gate": needles_all,
        "computed": ["any_needle", "none"],
        "tags": tags,
    }

    out_path.write_text(
        json.dumps(spec, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    return spec

def build_allowed_tags_and_defs_from_spec(spec: dict):
    if "tags" not in spec or not isinstance(spec["tags"], list):
        raise ValueError("Invalid TagSpec: missing 'tags' list")

    tag_list = []
    seen = set()
    for t in spec["tags"]:
        tag = t.get("tag")
        if not tag or not isinstance(tag, str):
            continue
        if tag in seen:
            continue
        seen.add(tag)
        tag_list.append(tag)

    computed = spec.get("computed", [])
    allowed = []
    if "any_needle" in computed:
        allowed.append("any_needle")
    allowed.extend(tag_list)
    if "none" in computed:
        allowed.append("none")

    defs = {}
    for t in spec["tags"]:
        tag = t.get("tag")
        if not tag:
            continue
        desc = (t.get("desc") or "").strip()
        defs[tag] = desc if desc else f"tag: {tag}"

    if "any_needle" in computed:
        defs["any_needle"] = "computed: at least one non-none tag applies (deterministic; not model-driven)"
    if "none" in computed:
        defs["none"] = "none of the above apply"

    return allowed, defs

if GENERATE_SPEC:
    spec = compile_tag_spec_from_needles_json(
        needles_json_path=NEEDLE_SOURCE_JSON,
        mode=NEEDLE_MODE,
        out_path=TAG_SPEC_PATH,
    )
    print("[tag_spec] generated from:", NEEDLE_SOURCE_JSON)
else:
    spec = load_tag_spec(TAG_SPEC_PATH)
    print("[tag_spec] loaded existing:", TAG_SPEC_PATH)

ALLOWED_TAGS, TAG_DEFS = build_allowed_tags_and_defs_from_spec(spec)

print("[mode]", NEEDLE_MODE)
print("[generate_spec]", GENERATE_SPEC)
print("[matches_root]", MATCHES_ROOT)
print("[needle_source_json]", NEEDLE_SOURCE_JSON)
print("[tag_spec_path]", TAG_SPEC_PATH)
print("[tag_spec] allowed tags:", ALLOWED_TAGS)
print("[out_enhanced_csv]", OUT_ENHANCED_CSV)
print("[out_y_json]", OUT_Y_JSON)

# =========================================================
# PROMPT BUILDER (now driven by TagSpec)
# =========================================================
def make_needle_prompt(text: str) -> str:
    # model should NOT output "any_needle"
    model_allowed = [t for t in ALLOWED_TAGS if t not in ("any_needle",)]
    tag_lines = "\n".join([f'- "{t}": {TAG_DEFS.get(t, "")}' for t in model_allowed])

    return f"""
TASK:
Given TEXT, select all applicable TAGS from the allowed list.

CRITICAL TAG RULE:
- Each "tag" value MUST be EXACTLY one of the strings in ALLOWED_TAGS (character-for-character).
- Do NOT invent new tag names.
- Do NOT output "any_needle" (it will be computed downstream).
- If uncertain, return ONLY ["none"].

ALLOWED_TAGS:
{tag_lines}

RULES:
- For every selected tag (except "none"), provide:
  - confidence in [0,1]
  - evidence_quote copied verbatim from TEXT (max 200 chars)
  - negated: true if TEXT explicitly indicates the opposite
- If you cannot quote evidence from TEXT, do NOT select that tag.
- Return VALID JSON ONLY. No markdown, no commentary, no extra keys.

TEXT:
<<<
{text.strip()}
>>>

OUTPUT JSON SCHEMA:
{{
  "selected": [
    {{"tag": "...", "confidence": 0.0, "negated": false, "evidence_quote": "..."}}
  ]
}}
""".strip()

# =========================================================
# ENGINE CONFIG
# =========================================================
cfg = YRunnerConfig(
    csv_path=CSV_PATH,
    text_col=TEXT_COL,
    out_dir=OUT_DIR,
    out_enhanced_csv=OUT_ENHANCED_CSV,
    out_y_json=OUT_Y_JSON,
    y_spec_py=Y_SPEC_PY,
    model=MODEL,
    debug=DEBUG,
    debug_slice=DEBUG_SLICE,
    needle_timeout=NEEDLE_TIMEOUT,
    y_spec_timeout=Y_SPEC_TIMEOUT,
    matches_root=MATCHES_ROOT,
    master_csv_name="_match_frequencies.csv",
    strict_schema_gate=True,
    schema_gate_source="tag_spec",
    tag_spec_path=TAG_SPEC_PATH,
)

df_out, y_results, diag = run_y_pipeline(
    cfg=cfg,
    allowed_tags=ALLOWED_TAGS,
    tag_defs=TAG_DEFS,
    needle_prompt_fn=make_needle_prompt,
    canonicalize_fn=canonicalize_tag,
)

display(df_out.head(25))
if diag is not None:
    display(diag.head(50))

In [2]:
import pandas as pd
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================

BASE_MATCHES_ROOT = Path("/media/hello/Vault/Tribunals/_Matches").resolve()

OFF_PATH = BASE_MATCHES_ROOT / "offensive" / "_matches_index.csv"
DEF_PATH = BASE_MATCHES_ROOT / "defensive" / "_matches_index.csv"

# =========================================================
# LOAD
# =========================================================

df_off = pd.read_csv(OFF_PATH)
df_def = pd.read_csv(DEF_PATH)

print(f"[offensive] rows: {len(df_off)}")
print(f"[defensive] rows: {len(df_def)}")

# =========================================================
# GET MATCH COLUMNS
# =========================================================

off_cols = [c for c in df_off.columns if c.startswith("has__")]
def_cols = [c for c in df_def.columns if c.startswith("has__")]

all_cols = sorted(set(off_cols) | set(def_cols))

# ensure all columns exist in both dfs
for c in all_cols:
    if c not in df_off:
        df_off[c] = False
    if c not in df_def:
        df_def[c] = False

# force boolean
for c in all_cols:
    df_off[c] = df_off[c].fillna(False).astype(bool)
    df_def[c] = df_def[c].fillna(False).astype(bool)

# =========================================================
# FREQUENCY CALCULATION
# =========================================================

rows = []

for col in all_cols:
    off_count = df_off[col].sum()
    def_count = df_def[col].sum()

    off_pct = (off_count / len(df_off)) * 100 if len(df_off) else 0
    def_pct = (def_count / len(df_def)) * 100 if len(df_def) else 0

    rows.append({
        "tag": col,
        "off_count": int(off_count),
        "off_pct": round(off_pct, 2),
        "def_count": int(def_count),
        "def_pct": round(def_pct, 2),
        "diff_pct": round(off_pct - def_pct, 2),
    })

freq_df = pd.DataFrame(rows).sort_values(
    by="diff_pct",
    ascending=False
).reset_index(drop=True)

print("\n=== OFFENSIVE vs DEFENSIVE ===\n")
print(freq_df)

# optional save
out_path = BASE_MATCHES_ROOT / "_match_frequencies_comparison.csv"
freq_df.to_csv(out_path, index=False)

print(f"\n[csv] written to: {out_path}")

[offensive] rows: 12994
[defensive] rows: 13181

=== OFFENSIVE vs DEFENSIVE ===

                                            tag  off_count  off_pct  \
0                                 has__evidence      12402    95.44   
1                                   has__appeal      10427    80.24   
2                                     has__role       8457    65.08   
3                            has__investigation       7037    54.16   
4                                has__grievance       6962    53.58   
5                                   has__duties       5560    42.79   
6                              has__performance       5337    41.07   
7                                  has__warning       4957    38.15   
8                     has__procedural_prejudice       3708    28.54   
9                             has__instructions       3470    26.70   
10                 has__management_acquiescence       3029    23.31   
11     has__predetermination_and_appeal_failure       1259     9.69

# MOLTIE – Jupyter Inference Cell (y_spec + Needle Runner)

## Purpose

This notebook cell executes the `y_spec.py` schema inference module **and** a closed-set needle classifier over each row of a witness statement CSV file. It is designed for **interactive debugging, enrichment, and calibration**, not bulk production execution.

It enables row-level inspection of semantic tagging (needles), structured Y inference, JSON validity, and stability before transitioning to a production batch workflow.

---

## Input

* **CSV file**:
  `/home/hello/Projects/Statements/input/Leonardo_WS.csv`

* **Column used**:
  `text_verbatim`

Each row from `text_verbatim` is treated as `ws_text` and:

1. Passed via `stdin` to `y_spec.py`
2. Independently evaluated by the needle classifier (LLM closed-set tagging)

---

## Outputs

Two files are written to:

`/home/hello/Projects/Statements/output`

1. **Leonardo_WS_enhanced.csv**

   * Contains **all original CSV columns**
   * Adds:

     * `needle_selected_raw`
     * `has__/conf__/quote__` columns per tag
     * `y_ok`, `y_rc`
     * `ws_len`
     * `X1` (1-based row id)
     * `doc`

2. **Y_inferred.json**

   * Aggregates structured `y_spec` output per processed row
   * Preserves full Y JSON for auditability

---

## Execution Flow

### 1. Configuration

The cell defines:

* Model name (`mistral-small3.2:latest`)
* Path to `y_spec.py`
* Debug mode toggle
* Flexible row selector (`DEBUG_SLICE`)
* Needle tag taxonomy (closed set)

Debug selector supports:

* `"3"` → process exactly the 3rd row (1-based)
* `":5"` → first 5 rows
* `"2:5"` → Python slice semantics

This allows precise surgical debugging.

---

### 2. CSV Loading

* Reads the CSV using pandas
* Validates `text_verbatim` exists
* Converts nulls to empty strings
* Determines which rows to process (full run or debug slice)

Original CSV structure is preserved.

---

### 3. Row Iteration (with tqdm)

For each selected row:

* Assigns a deterministic 1-based identifier (`X1`)
* Strips whitespace
* Skips empty rows
* Prints trace:

```
X1=<row_number> doc=Leonardo_WS.csv
```

This guarantees reproducibility and traceability.

---

### 4. Dual Processing Per Row

Each `ws_text` flows through two independent pipes:

#### A) Needle Classifier (Semantic Tagging)

* Closed-set LLM classification
* Evidence-quoted
* Negation-aware
* JSON-validated
* Returns:

  * Selected tags
  * Confidence scores
  * Evidence quotes

Expanded into structured columns (`has__/conf__/quote__`).

#### B) Y-Spec Structured Inference

```
python y_spec.py --model mistral-small3.2:latest
```

* Input via `stdin`
* Captures:

  * `stdout`
  * `stderr`
  * `returncode`
* Attempts immediate JSON parsing
* Aggregates valid results into `Y_inferred.json`

Needle tagging and Y inference remain structurally independent.

---

### 5. Enrichment Merge

The enrichment fields are merged back into the **full original DataFrame**, ensuring:

* No original data is lost
* Only processed rows receive populated enrichment fields (in debug mode)
* Non-processed rows remain intact

---

## Why This Design Is Intentional

This notebook version prioritises:

* Structural independence between retrieval (needles) and reasoning (Y)
* Full preservation of source data
* Evidence-anchored tagging
* Deterministic debug slicing
* Immediate JSON validation
* Transparent failure surfaces

It does **not** parallelise.

It does **not** bias X/Y extraction toward needle concepts.

It maintains clean architectural separation between semantic tagging and structured inference.

---

## When to Transition to Production Script

Move to a full `.py` batch runner once:

* Needle classification is stable
* Y JSON schema is consistent
* Debug slicing no longer required
* Error patterns are understood

Production version should then:

* Support parallel execution
* Implement retry logic
* Log failures explicitly
* Optionally write per-row Y JSON files

---

## Role in the MOLTIE Architecture

This cell functions as a **calibration and enrichment harness**.

It sits between:

* Raw witness statement substrate
* Independent semantic tagging layer
* Structured Y inference layer

It ensures that:

* Retrieval signals (needles)
* Structural reasoning signals (Y)

are derived independently from the same text.

---

## Summary

This Jupyter cell provides a controlled inference environment that:

* Reads witness statement rows
* Applies independent needle classification
* Executes `y_spec.py`
* Validates structured JSON output
* Merges enrichment into the original dataset
* Writes consolidated outputs

It is modular, auditable, deterministic under debug, and structurally clean.


In [ ]:
import json, re
from pathlib import Path

# =========================================================
# CONFIG
# =========================================================

BASE_MATCHES_ROOT = Path("/media/hello/Vault/Tribunals/_Matches").resolve()
NEEDLES_INPUT_ROOT = Path("/home/hello/Projects/Statements/input/needles").resolve()

NEEDLE_JSON_BY_MODE = {
    "offensive": NEEDLES_INPUT_ROOT / "offensive_needles.json",
    "defensive": NEEDLES_INPUT_ROOT / "defensive_needles.json",
}

# =========================================================
# HELPERS
# =========================================================

def canonicalize_tag(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def build_desc_for_keyword(tag: str) -> str:
    return f"keyword/phrase: '{tag}'"

def build_desc_for_regex(tag: str) -> str:
    return f"regex bucket: {tag.replace('_', ' ')}"

# =========================================================
# CORE BUILDER
# =========================================================

def compile_tag_spec_from_json(mode: str, df=None):
    if mode not in NEEDLE_JSON_BY_MODE:
        raise ValueError(f"Unsupported mode: {mode}")

    source_path = NEEDLE_JSON_BY_MODE[mode].resolve()
    matches_root = (BASE_MATCHES_ROOT / mode).resolve()
    matches_root.mkdir(parents=True, exist_ok=True)

    payload = json.loads(source_path.read_text(encoding="utf-8"))

    needles_all = payload.get("needles_all", [])
    needles_any = payload.get("needles_any", [])
    regex_buckets = payload.get("regex_buckets", {})

    reserved = {"any_needle", "none"}
    tags = []
    seen = set()

    # =====================================================
    # NEEDLES_ANY → TAGS
    # =====================================================
    for raw in needles_any:
        if not isinstance(raw, str):
            continue

        tag = canonicalize_tag(raw)
        if not tag or tag in reserved or tag in seen:
            continue

        seen.add(tag)
        tags.append({
            "tag": tag,
            "type": "needle_any",
            "source": {"phrase": raw},
            "desc": build_desc_for_keyword(raw),
            "corpus_column": f"has__{tag}",
        })

    # =====================================================
    # REGEX_BUCKETS → TAGS
    # =====================================================
    for bucket_name, meta in regex_buckets.items():
        if not isinstance(bucket_name, str):
            continue

        tag = canonicalize_tag(bucket_name)
        if not tag or tag in reserved or tag in seen:
            continue

        seen.add(tag)
        tags.append({
            "tag": tag,
            "type": "regex",
            "source": {
                "name": bucket_name,
                "pattern": meta.get("pattern"),
                "flags": meta.get("flags", []),
            },
            "desc": build_desc_for_regex(bucket_name),
            "corpus_column": f"has__{tag}",
        })

    # =====================================================
    # OPTIONAL: ATTACH PREVALENCE (if df passed)
    # =====================================================
    if df is not None and len(df) > 0:
        n = len(df)
        cols = set(df.columns)

        for t in tags:
            col = t["corpus_column"]

            if col in cols:
                s = df[col].fillna(False).astype(bool)
                cnt = int(s.sum())
                pct = (cnt / n) * 100.0

                t["stats"] = {
                    "count": cnt,
                    "pct": round(pct, 2),
                    "n_cases": n
                }

                t["desc"] += f" (hits {cnt}/{n}; {pct:.2f}%)"
            else:
                t["stats"] = {
                    "count": None,
                    "pct": None,
                    "n_cases": n
                }

    # =====================================================
    # FINAL SPEC
    # =====================================================
    spec = {
        "version": "v2",
        "compiled_from": str(source_path),
        "mode": mode,
        "needles_all_gate": [canonicalize_tag(x) for x in needles_all],
        "computed": ["any_needle", "none"],
        "tags": tags,
    }

    out_path = matches_root / "_tag_spec.json"
    out_path.write_text(
        json.dumps(spec, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    return out_path

# =========================================================
# RUN (BOTH MODES)
# =========================================================

for mode in ["offensive", "defensive"]:
    out = compile_tag_spec_from_json(mode=mode)
    print(f"[tag_spec] {mode} → {out}")

In [5]:
import json
from pathlib import Path

# =========================================================
# PATHS
# =========================================================

BASE_MATCHES_ROOT = Path("/media/hello/Vault/Tribunals/_Matches").resolve()

TAG_SPEC_PATHS = {
    "offensive": (BASE_MATCHES_ROOT / "offensive" / "_tag_spec.json").resolve(),
    "defensive": (BASE_MATCHES_ROOT / "defensive" / "_tag_spec.json").resolve(),
}

# =========================================================
# HELPERS
# =========================================================

def load_tag_spec(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"TagSpec not found: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def build_vocab_from_tag_spec(spec: dict):
    if "tags" not in spec or not isinstance(spec["tags"], list):
        raise ValueError("Invalid TagSpec: missing 'tags' list")

    tags = [t["tag"] for t in spec["tags"] if isinstance(t, dict) and t.get("tag")]

    # preserve order, unique
    seen = set()
    tags_u = []
    for t in tags:
        if t in seen:
            continue
        seen.add(t)
        tags_u.append(t)

    allowed = []
    if "any_needle" in spec.get("computed", []):
        allowed.append("any_needle")
    allowed.extend(tags_u)
    if "none" in spec.get("computed", []):
        allowed.append("none")

    defs = {}
    for t in spec["tags"]:
        tag = t.get("tag")
        if not tag:
            continue
        defs[tag] = (t.get("desc") or f"tag: {tag}")

    if "any_needle" in spec.get("computed", []):
        defs["any_needle"] = (
            "computed: at least one non-none tag applies "
            "(deterministic; not model-driven)"
        )
    if "none" in spec.get("computed", []):
        defs["none"] = "none of the above apply"

    return allowed, defs

# =========================================================
# LOAD BOTH SPECS
# =========================================================

VOCABS = {}

for mode, path in TAG_SPEC_PATHS.items():
    spec = load_tag_spec(path)
    allowed_tags, tag_defs = build_vocab_from_tag_spec(spec)

    VOCABS[mode] = {
        "path": path,
        "spec": spec,
        "allowed_tags": allowed_tags,
        "tag_defs": tag_defs,
    }

    print(f"[vocab] {mode} loaded from: {path}")
    print(f"[vocab] {mode} allowed tags ({len(allowed_tags)}): {allowed_tags}\n")

# =========================================================
# CONVENIENCE VARIABLES
# =========================================================

OFF_SPEC = VOCABS["offensive"]["spec"]
OFF_ALLOWED_TAGS = VOCABS["offensive"]["allowed_tags"]
OFF_TAG_DEFS = VOCABS["offensive"]["tag_defs"]

DEF_SPEC = VOCABS["defensive"]["spec"]
DEF_ALLOWED_TAGS = VOCABS["defensive"]["allowed_tags"]
DEF_TAG_DEFS = VOCABS["defensive"]["tag_defs"]

[vocab] offensive loaded from: /media/hello/Vault/Tribunals/_Matches/offensive/_tag_spec.json
[vocab] offensive allowed tags (23): ['any_needle', 'no_contemporaneous_evidence', 'predetermination', 'gross_misconduct', 'disciplinary_hearing', 'investigation', 'appeal', 'grievance', 'role', 'duties', 'instructions', 'performance', 'evidence', 'warning', 'undefined_primary_duty', 'absence_of_contemporaneous_prohibition', 'management_acquiescence', 'post_hoc_intent_inference', 'logical_inconsistency_performance', 'failure_to_engage_core_defence', 'procedural_prejudice', 'predetermination_and_appeal_failure', 'none']

[vocab] defensive loaded from: /media/hello/Vault/Tribunals/_Matches/defensive/_tag_spec.json
[vocab] defensive allowed tags (19): ['any_needle', 'gross_misconduct', 'reasonable_investigation', 'disciplinary_hearing', 'policy', 'trust_and_confidence', 'misconduct', 'dismissal', 'queue', 'tickets', 'system', 'intentional_misconduct_pattern', 'policy_breach_queue_priority', 'reas